# Risk Metric Diagnosis and Evaluation

These queries describe the current real public ingestion and then evaluate the deterministic engine. They do not estimate internal platform violation prevalence. Reads the committed snapshot and the project's own evaluation functions, so results reproduce from a clean checkout.

In [1]:
import sys, json, duckdb, pandas as pd
sys.path.insert(0, "..")
pd.set_option("display.width", 140)
db = duckdb.connect("../data/deploy/adshield.duckdb", read_only=True)

## 1. Language and routing mix

What languages appear, and where do cases route? CFPB priors can only route to research actions — never ad enforcement.

In [2]:
print(db.execute("SELECT language, count(*) AS cases, round(avg(CAST(risk_score>=0.65 AS INT)),3) AS high_priority_rate FROM ad_risk_scores GROUP BY 1 ORDER BY 2 DESC").df().to_string(index=False))
print()
print(db.execute("SELECT recommended_action, count(*) AS cases FROM ad_risk_scores GROUP BY 1 ORDER BY 2 DESC").df().to_string(index=False))

language  cases  high_priority_rate
      en    956               0.003

           recommended_action  cases
            use as risk prior    953
prioritize for analyst review      3


## 2. Threshold sweep: review load vs missed risk

Uses the project's own `threshold_sensitivity` over the curated scenarios. Lowering the escalation threshold surfaces more cases (review load) but reduces missed high-severity scenarios — the core operating trade-off, quantified rather than asserted.

In [3]:
from src.risk.lifecycle import threshold_sensitivity
ts = pd.DataFrame(threshold_sensitivity(capacity=120))
print(ts[["threshold","review_volume","missed_risk_scenarios","false_positive_scenarios","capacity_utilization"]].to_string(index=False))

 threshold  review_volume  missed_risk_scenarios  false_positive_scenarios  capacity_utilization
      0.35             49                      0                         0                 0.408
      0.45             49                      0                         0                 0.408
      0.55             49                      0                         0                 0.408
      0.65             49                      0                         0                 0.408
      0.69             49                      0                         0                 0.408


## 3. External validation: does human perception match the taxonomy?

The UW CHI 2021 dataset is 500 real web ads rated by 1,025 independent participants. The ads are **image screenshots with no ad copy**, so the text engine cannot score them and no per-ad engine-vs-human agreement is claimed. Instead we ask a category-level question: do independent humans perceive the most deception in the same areas the taxonomy prioritizes?

In [4]:
uw = json.load(open("../data/public_validation/uw_bad_ads_summary.json"))
pc = pd.DataFrame(uw["perception_by_content_category"])
print(pc[["content_category","ad_count","mean_deceptive_share","mean_clickbait_share","mapped_risk_category"]].head(8).to_string(index=False))

         content_category  ad_count  mean_deceptive_share  mean_clickbait_share                        mapped_risk_category
                 Listicle        29                0.3395                0.6981               Deceptive / Misleading Claims
   Health and Supplements        29                0.4455                0.5727 Health / Weight Loss / Pharmaceuticals Risk
              Advertorial        42                0.3836                0.5538               Deceptive / Misleading Claims
                   Native        90                0.3416                0.5713               Deceptive / Misleading Claims
        Software Download        24                0.4155                0.3348                                         NaN
         Sponsored Search        16                0.2153                0.3227               Deceptive / Misleading Claims
Computer Security-related        15                0.2664                0.2309                                         NaN
        

**Finding.** The highest independently-perceived deception/clickbait sits in Health & Supplements and in advertorial/listicle/native formats — the taxonomy's health and deceptive-claims categories. Low-perceived-deception categories (journalism, apparel, B2B) are not prioritized. This is category-level external corroboration of the risk ranking, not per-ad enforcement agreement.

## 4. Generalization: development set vs held-out

The 60 curated scenarios are the set candidate v2.1 was tuned against. An 18-scenario held-out set, authored after v2.1 was frozen, estimates generalization. The gap is the honest cost of tuning against a fixed label set.

In [5]:
from src.app.repository import holdout_benchmark
h = holdout_benchmark()
rows = [
  ("development (60, tuned)", "v1",   h["development_set"]["v1"]["category_agreement"],           h["development_set"]["v1"]["routing_agreement"]),
  ("development (60, tuned)", "v2.1", h["development_set"]["candidate_v2_1"]["category_agreement"], h["development_set"]["candidate_v2_1"]["routing_agreement"]),
  ("held-out (18, unseen)",  "v1",   h["holdout_set"]["v1"]["category_agreement"],               h["holdout_set"]["v1"]["routing_agreement"]),
  ("held-out (18, unseen)",  "v2.1", h["holdout_set"]["candidate_v2_1"]["category_agreement"],     h["holdout_set"]["candidate_v2_1"]["routing_agreement"]),
]
print(pd.DataFrame(rows, columns=["set","strategy","category_agreement","routing_agreement"]).to_string(index=False))
print()
print("category generalization gap (v2.1):", h["generalization_gap"]["category_agreement"])
print("routing  generalization gap (v2.1):", h["generalization_gap"]["routing_agreement"])

                    set strategy  category_agreement  routing_agreement
development (60, tuned)       v1               0.500              0.667
development (60, tuned)     v2.1               0.933              1.000
  held-out (18, unseen)       v1               0.278              0.556
  held-out (18, unseen)     v2.1               0.500              0.722

category generalization gap (v2.1): 0.433
routing  generalization gap (v2.1): 0.278


**Conclusion.** Candidate v2.1 drops sharply from the development set to the held-out set, so the development-set score is regression coverage, not generalization. It still beats v1 on unseen scenarios, so the improvements carry real signal. Most held-out misses are novel phrasings the keyword matcher has no term for — the documented coverage limit, made concrete.